# Comment Letter Evaluation with OpenAI

This notebook loads comment JSON files one at a time and asks the OpenAI API to score each comment on relevance, reasoning, evidence, impact, actionability, and structure/formatting.

Before running, set `OPENAI_API_KEY` in your environment. Results are saved to JSONL and CSV in the `LLM-eval` folder.

In [15]:
# Install dependencies (run once)
%pip install -q openai jsonschema pandas tqdm python-dotenv

Note: you may need to restart the kernel to use updated packages.


## Configure OpenAI API Client

This section loads the API key from the environment and sets default request settings.

In [16]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is not set in the environment.")

client = OpenAI(api_key=api_key)

MODEL = "gpt-5.4-mini"
TEMPERATURE = 0.2
MAX_TOKENS = 600

## Define Evaluation Criteria and Scoring Schema

Scores are 1 to 5 for each criterion, with short justifications. The schema enforces a strict JSON response.

In [17]:
from jsonschema import validate, Draft7Validator

RUBRIC = {
    "relevance": "How directly the comment addresses the docket topic and requested issues.",
    "reasoning": "Logical coherence and clarity of arguments.",
    "evidence": "Use of facts, data, citations, or concrete examples.",
    "impact": "Specificity and plausibility of claimed impacts.",
    "actionability": "Provides clear, feasible recommendations or requests.",
    "structure_formatting": "Organization, readability, and professional formatting.",
    "overall": "Holistic quality across all criteria."
}

RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "scores": {
            "type": "object",
            "properties": {
                "relevance": {"type": "integer", "minimum": 1, "maximum": 5},
                "reasoning": {"type": "integer", "minimum": 1, "maximum": 5},
                "evidence": {"type": "integer", "minimum": 1, "maximum": 5},
                "impact": {"type": "integer", "minimum": 1, "maximum": 5},
                "actionability": {"type": "integer", "minimum": 1, "maximum": 5},
                "structure_formatting": {"type": "integer", "minimum": 1, "maximum": 5},
                "overall": {"type": "integer", "minimum": 1, "maximum": 5}
            },
            "required": [
                "relevance",
                "reasoning",
                "evidence",
                "impact",
                "actionability",
                "structure_formatting",
                "overall"
            ]
        },
        "rationales": {
            "type": "object",
            "properties": {
                "relevance": {"type": "string"},
                "reasoning": {"type": "string"},
                "evidence": {"type": "string"},
                "impact": {"type": "string"},
                "actionability": {"type": "string"},
                "structure_formatting": {"type": "string"},
                "overall": {"type": "string"}
            },
            "required": [
                "relevance",
                "reasoning",
                "evidence",
                "impact",
                "actionability",
                "structure_formatting",
                "overall"
            ]
        },
        "overall_summary": {"type": "string"}
    },
    "required": ["scores", "rationales", "overall_summary"]
}

_validator = Draft7Validator(RESPONSE_SCHEMA)


def validate_response(data):
    errors = sorted(_validator.iter_errors(data), key=lambda e: e.path)
    if errors:
        raise ValueError("Schema validation failed: " + "; ".join([e.message for e in errors]))
    return data

## Load Comment JSON Files from LLM-eval

Set `COMMENTS_DIR` to the folder that contains comment JSON files. By default it points to `LLM-eval`.

In [18]:
import html
import json
import re
from pathlib import Path

BASE_DIR = Path("/Users/norazhan/Desktop/MDI/Statt")
COMMENTS_DIR = BASE_DIR / "LLM-eval" / "comments"
CONTEXTS_PATH = BASE_DIR / "LLM-eval" / "policy_contexts.json"


def list_json_files(folder: Path):
    return sorted([p for p in folder.rglob("*.json") if p.is_file()])


def normalize_text(text):
    text = html.unescape(text)
    text = text.replace("<br/>", "\n").replace("<br>", "\n").replace("<br />", "\n")
    return text.strip()


def extract_comment_text(payload):
    # Nested structure used in these datasets
    text_obj = payload.get("text")
    if isinstance(text_obj, dict):
        full_text = text_obj.get("fullText")
        if isinstance(full_text, str) and full_text.strip():
            return normalize_text(full_text)

    # Try common flat fields
    candidate_keys = [
        "comment",
        "comment_text",
        "commentText",
        "text",
        "content",
        "body"
    ]
    for key in candidate_keys:
        value = payload.get(key)
        if isinstance(value, str) and value.strip():
            return normalize_text(value)

    # Fallback: concatenate string values
    text_parts = [v for v in payload.values() if isinstance(v, str) and v.strip()]
    if text_parts:
        return normalize_text("\n\n".join(text_parts))
    return None


def extract_policy_id(payload, comment_text, path: Path):
    # Prefer explicit metadata if present
    for key in ["docketId", "docket_id", "docketID", "docket", "policy_id", "policyId"]:
        value = payload.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()

    # Try to infer from comment text
    if isinstance(comment_text, str):
        patterns = [
            r"\bEPA-[A-Z]{2}-[A-Z]{2}-\d{4}-\d{4}\b",
            r"\bFDA-\d{4}-[A-Z]-\d{4}\b",
            r"\bFMCSA-\d{4}-\d{4}\b",
            r"\b[A-Z]{3,}-\d{4}-[A-Z]-\d{3,5}\b",
            r"\b[A-Z]{3,}-\d{4}-\d{3,6}\b"
        ]
        for pattern in patterns:
            match = re.search(pattern, comment_text)
            if match:
                return match.group(0)

    return "unknown"


def extract_source_label(payload, path: Path):
    # Keep labels for analysis, but do not pass them to the API
    for key in ["source_label", "author_type", "label"]:
        value = payload.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip().lower()
    for part in path.parts:
        if part.lower() in {"human", "ai"}:
            return part.lower()
    return "unknown"


def load_policy_contexts(path: Path):
    if not path.exists():
        return {}
    data = json.loads(path.read_text(encoding="utf-8"))
    return data if isinstance(data, dict) else {}


policy_contexts = load_policy_contexts(CONTEXTS_PATH)
json_files = list_json_files(COMMENTS_DIR)
len(json_files)

300

## Build Prompt and Call the OpenAI API

The prompt requests a JSON-only response that follows the schema.

In [19]:
import textwrap


def build_prompt(comment_text, policy_id=None, policy_context=None):
    rubric_lines = "\n".join([f"- {k}: {v}" for k, v in RUBRIC.items()])
    policy_block = ""
    if policy_id or policy_context:
        policy_block = textwrap.dedent(
            f"""
            Policy context:
            - Policy ID: {policy_id or "unknown"}
            - Summary: {policy_context or "(no summary provided)"}
            """
        )

    return textwrap.dedent(
        f"""
        You are evaluating a public comment letter. Score each criterion from 1 (poor) to 5 (excellent).
        Provide a short rationale per criterion and a brief overall summary.

        Criteria:
        {rubric_lines}

        {policy_block}
        Return ONLY valid JSON with this structure:
        {{
          "scores": {{"relevance": 1-5, "reasoning": 1-5, "evidence": 1-5, "impact": 1-5,
                      "actionability": 1-5, "structure_formatting": 1-5, "overall": 1-5}},
          "rationales": {{"relevance": "...", "reasoning": "...", "evidence": "...", "impact": "...",
                         "actionability": "...", "structure_formatting": "...", "overall": "..."}},
          "overall_summary": "..."
        }}

        Comment:
        """ + comment_text.strip()
    )


def call_openai(comment_text, policy_id=None, policy_context=None):
    prompt = build_prompt(comment_text, policy_id=policy_id, policy_context=policy_context)
    response = client.responses.create(
        model=MODEL,
        input=prompt,
        temperature=TEMPERATURE,
        max_output_tokens=MAX_TOKENS,
    )
    return response.output_text

## Parse Model Response and Validate JSON Output

This attempts to extract JSON if the model returns extra text, then validates the schema.

In [20]:
import re


def extract_json(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in response")
    return json.loads(match.group(0))


def evaluate_comment(comment_text, policy_id=None, policy_context=None):
    raw = call_openai(comment_text, policy_id=policy_id, policy_context=policy_context)
    data = extract_json(raw)
    validate_response(data)
    return data

## Process Files One-by-One and Save Results

This runs through files, collects results, and writes JSONL and CSV outputs.

In [21]:
import pandas as pd
from tqdm import tqdm

OUTPUT_JSONL = BASE_DIR / "LLM-eval" / "comment_scores.jsonl"
OUTPUT_CSV = BASE_DIR / "LLM-eval" / "comment_scores.csv"

MAX_FILES = None  # set to an integer for a smaller test run
RESUME = False  # skip already-scored files in OUTPUT_JSONL

results = []
processed = set()

if RESUME and OUTPUT_JSONL.exists():
    for line in OUTPUT_JSONL.read_text(encoding="utf-8").splitlines():
        try:
            row = json.loads(line)
            if isinstance(row, dict) and "file" in row:
                results.append(row)
                processed.add(row["file"])
        except json.JSONDecodeError:
            continue

for path in tqdm(json_files[:MAX_FILES] if MAX_FILES else json_files):
    if RESUME and str(path) in processed:
        continue
    try:
        with path.open("r", encoding="utf-8") as f:
            payload = json.load(f)
        comment_text = extract_comment_text(payload)
        source_label = extract_source_label(payload, path)
        policy_id = extract_policy_id(payload, comment_text, path)
        policy_context = policy_contexts.get(policy_id, "")
        if not comment_text:
            results.append({
                "file": str(path),
                "policy_id": policy_id,
                "source_label": source_label,
                "error": "No comment text found"
            })
            continue
        evaluation = evaluate_comment(
            comment_text,
            policy_id=policy_id,
            policy_context=policy_context
        )
        results.append({
            "file": str(path),
            "policy_id": policy_id,
            "source_label": source_label,
            "comment_text": comment_text,
            **evaluation
        })
    except Exception as exc:
        results.append({
            "file": str(path),
            "policy_id": "unknown",
            "source_label": "unknown",
            "error": str(exc)
        })

with OUTPUT_JSONL.open("w", encoding="utf-8") as f:
    for row in results:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def sanitize_csv_field(value):
    if value is None:
        return ""
    if not isinstance(value, str):
        return value
    # Keep CSV rows single-line for spreadsheet tools
    cleaned = value.replace("\r\n", " ").replace("\n", " ").replace("\r", " ")
    return " ".join(cleaned.split())


# Flatten scores/rationales for CSV
flat_rows = []
for row in results:
    if "scores" in row:
        flat = {
            "file": row["file"],
            "policy_id": row.get("policy_id", "unknown"),
            "source_label": row.get("source_label", "unknown"),
            "comment_text": sanitize_csv_field(row.get("comment_text", "")),
            "overall_summary": sanitize_csv_field(row.get("overall_summary", ""))
        }
        for k, v in row["scores"].items():
            flat[f"score_{k}"] = v
        for k, v in row["rationales"].items():
            flat[f"rationale_{k}"] = sanitize_csv_field(v)
        flat_rows.append(flat)
    else:
        flat_rows.append({
            "file": row["file"],
            "policy_id": row.get("policy_id", "unknown"),
            "source_label": row.get("source_label", "unknown"),
            "error": sanitize_csv_field(row.get("error", "Unknown error"))
        })

pd.DataFrame(flat_rows).to_csv(OUTPUT_CSV, index=False)

OUTPUT_JSONL, OUTPUT_CSV

100%|██████████| 300/300 [18:09<00:00,  3.63s/it]


(PosixPath('/Users/norazhan/Desktop/MDI/Statt/LLM-eval/comment_scores.jsonl'),
 PosixPath('/Users/norazhan/Desktop/MDI/Statt/LLM-eval/comment_scores.csv'))